# Lab 06 — 01 Gold Dimensions (External Tables, Serverless-Compatible)

**Dataset:** Synthea Healthcare  
**Target:** External Delta tables in a dedicated Gold schema  
**Compute:** Databricks Serverless compatible

## Purpose

Build the six Lab 06 Gold dimensions as **external Delta tables**:

- `dim_date`
- `dim_patient`
- `dim_provider`
- `dim_organization`
- `dim_payer`
- `dim_condition`

This version intentionally does **not** execute `REFRESH TABLE`, `CACHE TABLE`,
or other cache-management commands that are unsupported on Databricks Serverless.

The notebook is designed to work both:

- from `lab06_00_dev_runner` through `dbutils.notebook.run(...)`;
- interactively, after supplying the same parameters.


## 1. Runtime parameters

In [ ]:
def ensure_text_widget(name: str, default: str, label: str) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.text(name, default, label)


def ensure_dropdown_widget(
    name: str,
    default: str,
    choices: list[str],
    label: str,
) -> None:
    try:
        dbutils.widgets.get(name)
    except Exception:
        dbutils.widgets.dropdown(name, default, choices, label)


ensure_text_widget("catalog", "dbr_dev", "01 Catalog")
ensure_text_widget("source_schema", "parvinbadalov", "02 Source schema")
ensure_text_widget(
    "source_volume_name",
    "lab06_gold_analytics",
    "03 Source volume",
)
ensure_text_widget(
    "target_schema",
    "parvinbadalov_lab06_ext",
    "04 Target schema",
)
ensure_text_widget(
    "external_gold_root",
    "REPLACE_WITH_EXTERNAL_GOLD_ROOT",
    "05 External Gold root",
)
ensure_text_widget("date_start", "1900-01-01", "06 Date start")
ensure_text_widget("date_end", "2035-12-31", "07 Date end")
ensure_dropdown_widget(
    "rebuild_dim_date",
    "false",
    ["false", "true"],
    "08 Rebuild dim_date",
)
ensure_dropdown_widget(
    "run_validation",
    "true",
    ["true", "false"],
    "09 Run validation",
)

catalog = dbutils.widgets.get("catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_volume_name = dbutils.widgets.get("source_volume_name").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
external_gold_root = dbutils.widgets.get("external_gold_root").strip().rstrip("/")
date_start = dbutils.widgets.get("date_start").strip()
date_end = dbutils.widgets.get("date_end").strip()
rebuild_dim_date = (
    dbutils.widgets.get("rebuild_dim_date").strip().lower() == "true"
)
run_validation = (
    dbutils.widgets.get("run_validation").strip().lower() == "true"
)

print(f"Catalog            : {catalog}")
print(f"Source schema      : {source_schema}")
print(f"Source volume      : {source_volume_name}")
print(f"Target schema      : {target_schema}")
print(f"External Gold root : {external_gold_root}")
print(f"Date range         : {date_start} -> {date_end}")
print(f"Rebuild dim_date   : {rebuild_dim_date}")
print(f"Run validation     : {run_validation}")

## 2. Validate parameter values and resolve paths

In [ ]:
import re
from datetime import date

for parameter_name, value in {
    "catalog": catalog,
    "source_schema": source_schema,
    "source_volume_name": source_volume_name,
    "target_schema": target_schema,
}.items():
    if not re.fullmatch(r"[A-Za-z0-9_]+", value):
        raise ValueError(
            f"{parameter_name} contains unsupported characters: {value!r}"
        )

if (
    not external_gold_root
    or external_gold_root == "REPLACE_WITH_EXTERNAL_GOLD_ROOT"
):
    raise ValueError(
        "external_gold_root must be supplied by lab06_00_dev_runner "
        "or entered manually."
    )

if not external_gold_root.lower().startswith("abfss://"):
    raise ValueError(
        "external_gold_root must be an abfss:// Azure storage path."
    )

try:
    parsed_start = date.fromisoformat(date_start)
    parsed_end = date.fromisoformat(date_end)
except ValueError as exc:
    raise ValueError(
        "date_start and date_end must use YYYY-MM-DD format."
    ) from exc

if parsed_start > parsed_end:
    raise ValueError("date_start must be <= date_end.")

source_volume_fqn = (
    f"{catalog}.{source_schema}.{source_volume_name}"
)
source_volume_path = (
    f"/Volumes/{catalog}/{source_schema}/{source_volume_name}"
)
reference_path = f"{source_volume_path}/reference"

target_schema_fqn = f"{catalog}.{target_schema}"

TABLE_NAMES = {
    "date": f"{target_schema_fqn}.dim_date",
    "patient": f"{target_schema_fqn}.dim_patient",
    "provider": f"{target_schema_fqn}.dim_provider",
    "organization": f"{target_schema_fqn}.dim_organization",
    "payer": f"{target_schema_fqn}.dim_payer",
    "condition": f"{target_schema_fqn}.dim_condition",
}

TABLE_LOCATIONS = {
    logical_name: f"{external_gold_root}/{table_name.split('.')[-1]}"
    for logical_name, table_name in TABLE_NAMES.items()
}

print(f"Source volume path : {source_volume_path}")
print(f"Reference path     : {reference_path}")
print("")
print("External Gold targets:")
for logical_name in TABLE_NAMES:
    print(
        f"  {TABLE_NAMES[logical_name]} -> "
        f"{TABLE_LOCATIONS[logical_name]}"
    )

## 3. Validate source volume and staged reference data

In [ ]:
volume_df = spark.sql(
    f"DESCRIBE VOLUME {source_volume_fqn}"
)

display(volume_df)

volume_metadata = volume_df.first().asDict()
volume_type = str(volume_metadata.get("volume_type", "")).upper()

if volume_type != "EXTERNAL":
    raise RuntimeError(
        f"{source_volume_fqn} must be an EXTERNAL volume; "
        f"found {volume_type or 'UNKNOWN'}."
    )

REQUIRED_REFERENCE_FILES = {
    "patients.csv",
    "providers.csv",
    "organizations.csv",
    "payers.csv",
    "conditions.csv",
}

available_reference_files = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(reference_path)
}

missing_reference_files = sorted(
    REQUIRED_REFERENCE_FILES - available_reference_files
)

if missing_reference_files:
    raise FileNotFoundError(
        "Missing staged reference files: "
        + ", ".join(missing_reference_files)
    )

print("Source volume and reference files are ready.")

## 4. Create the target schema

The schema provides the Unity Catalog namespace.  
Each table is still **external** because its Delta data is stored at the explicit
`external_gold_root` location.


In [ ]:
spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {target_schema_fqn}"
)

print(f"Target schema ready: {target_schema_fqn}")

## 5. Serverless-safe external Delta helpers

In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def read_reference_csv(file_name: str) -> DataFrame:
    """Read a staged Synthea reference CSV as strings."""
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(f"{reference_path}/{file_name}")
    )


def normalize_location(value: str) -> str:
    return value.rstrip("/").lower()


def registered_table_location(table_name: str) -> str | None:
    """Return a registered Delta table location when the table exists."""
    if not spark.catalog.tableExists(table_name):
        return None

    detail = spark.sql(f"DESCRIBE DETAIL {table_name}").first().asDict()
    return detail.get("location")


def write_external_delta(
    df: DataFrame,
    table_name: str,
    location: str,
) -> None:
    """
    Idempotently overwrite an external Delta table.

    Serverless note:
    - writes Delta files directly to the requested external location;
    - registers the table with USING DELTA LOCATION when needed;
    - deliberately does NOT call REFRESH TABLE or cache APIs.
    """
    existing_location = registered_table_location(table_name)

    if existing_location is not None:
        if normalize_location(existing_location) != normalize_location(location):
            raise RuntimeError(
                f"{table_name} is already registered at "
                f"{existing_location}, but this run expects {location}."
            )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(location)
    )

    if not spark.catalog.tableExists(table_name):
        escaped_location = location.replace("'", "''")

        spark.sql(
            f"""
            CREATE TABLE {table_name}
            USING DELTA
            LOCATION '{escaped_location}'
            """
        )

    # IMPORTANT:
    # No REFRESH TABLE here. It is unsupported on Serverless and unnecessary
    # for this Delta write/register workflow.

    actual_location = registered_table_location(table_name)

    if (
        actual_location is None
        or normalize_location(actual_location)
        != normalize_location(location)
    ):
        raise RuntimeError(
            f"External registration validation failed for {table_name}. "
            f"Expected={location}; actual={actual_location}"
        )


def dimension_key_profile(
    table_name: str,
    key_column: str,
) -> tuple[int, int, int]:
    df = spark.table(table_name)

    row = (
        df.agg(
            F.count("*").alias("row_count"),
            F.countDistinct(key_column).alias("distinct_key_count"),
            F.sum(
                F.when(F.col(key_column).isNull(), 1).otherwise(0)
            ).alias("null_key_count"),
        )
        .first()
    )

    return (
        int(row["row_count"]),
        int(row["distinct_key_count"]),
        int(row["null_key_count"] or 0),
    )


print("Serverless-compatible external Delta helpers loaded.")

## 6. Build `dim_date`

In [ ]:
dim_date_exists = spark.catalog.tableExists(TABLE_NAMES["date"])

if (not dim_date_exists) or rebuild_dim_date:
    date_seed_df = spark.createDataFrame(
        [(date_start, date_end)],
        ["start_date", "end_date"],
    )

    dim_date_df = (
        date_seed_df
        .select(
            F.explode(
                F.sequence(
                    F.to_date("start_date"),
                    F.to_date("end_date"),
                    F.expr("INTERVAL 1 DAY"),
                )
            ).alias("full_date")
        )
        .select(
            F.date_format("full_date", "yyyyMMdd")
                .cast("int")
                .alias("date_key"),
            F.col("full_date"),
            F.year("full_date").alias("year"),
            F.quarter("full_date").alias("quarter"),
            F.month("full_date").alias("month"),
            F.date_format("full_date", "MMMM").alias("month_name"),
            F.weekofyear("full_date").alias("week_of_year"),
            F.dayofmonth("full_date").alias("day_of_month"),
            F.date_format("full_date", "EEEE").alias("day_name"),
            (
                F.pmod(
                    F.dayofweek("full_date") + F.lit(5),
                    F.lit(7),
                )
                + F.lit(1)
            ).alias("iso_day_of_week"),
            F.date_format("full_date", "yyyy-MM").alias("year_month"),
            F.when(
                F.dayofweek("full_date").isin(1, 7),
                F.lit(True),
            )
            .otherwise(F.lit(False))
            .alias("is_weekend"),
        )
    )

    write_external_delta(
        dim_date_df,
        TABLE_NAMES["date"],
        TABLE_LOCATIONS["date"],
    )

    print(f"Built {TABLE_NAMES['date']}")
else:
    existing_location = registered_table_location(TABLE_NAMES["date"])

    if normalize_location(existing_location) != normalize_location(
        TABLE_LOCATIONS["date"]
    ):
        raise RuntimeError(
            "Existing dim_date is registered at a different location. "
            "Set rebuild_dim_date=true only after correcting the target."
        )

    print(
        "dim_date already exists at the expected external location. "
        "Reusing it."
    )

display(
    spark.table(TABLE_NAMES["date"])
    .orderBy("full_date")
    .limit(10)
)

## 7. Build `dim_patient`

In [ ]:
patients_src = read_reference_csv("patients.csv")

dim_patient_df = (
    patients_src
    .select(
        F.xxhash64("Id").alias("patient_key"),
        F.col("Id").alias("patient_id"),
        F.to_date("BIRTHDATE").alias("birth_date"),
        F.to_date("DEATHDATE").alias("death_date"),
        F.col("SSN").alias("ssn"),
        F.col("FIRST").alias("first_name"),
        F.col("LAST").alias("last_name"),
        F.col("MAIDEN").alias("maiden_name"),
        F.col("MARITAL").alias("marital_status"),
        F.col("RACE").alias("race"),
        F.col("ETHNICITY").alias("ethnicity"),
        F.col("GENDER").alias("gender"),
        F.col("BIRTHPLACE").alias("birthplace"),
        F.col("ADDRESS").alias("address"),
        F.col("CITY").alias("city"),
        F.col("STATE").alias("state"),
        F.col("COUNTY").alias("county"),
        F.col("ZIP").alias("zip_code"),
        F.col("LAT").cast("double").alias("latitude"),
        F.col("LON").cast("double").alias("longitude"),
        F.col("HEALTHCARE_EXPENSES")
            .cast("decimal(18,2)")
            .alias("healthcare_expenses"),
        F.col("HEALTHCARE_COVERAGE")
            .cast("decimal(18,2)")
            .alias("healthcare_coverage"),
    )
    .dropDuplicates(["patient_id"])
)

write_external_delta(
    dim_patient_df,
    TABLE_NAMES["patient"],
    TABLE_LOCATIONS["patient"],
)

print(f"Built {TABLE_NAMES['patient']}")

## 8. Build `dim_organization`

In [ ]:
organizations_src = read_reference_csv("organizations.csv")

dim_organization_df = (
    organizations_src
    .select(
        F.xxhash64("Id").alias("organization_key"),
        F.col("Id").alias("organization_id"),
        F.col("NAME").alias("organization_name"),
        F.col("ADDRESS").alias("address"),
        F.col("CITY").alias("city"),
        F.col("STATE").alias("state"),
        F.col("ZIP").alias("zip_code"),
        F.col("LAT").cast("double").alias("latitude"),
        F.col("LON").cast("double").alias("longitude"),
        F.col("PHONE").alias("phone"),
        F.col("REVENUE")
            .cast("decimal(18,2)")
            .alias("revenue"),
        F.col("UTILIZATION")
            .cast("long")
            .alias("utilization"),
    )
    .dropDuplicates(["organization_id"])
)

write_external_delta(
    dim_organization_df,
    TABLE_NAMES["organization"],
    TABLE_LOCATIONS["organization"],
)

print(f"Built {TABLE_NAMES['organization']}")

## 9. Build `dim_provider`

In [ ]:
providers_src = read_reference_csv("providers.csv")

dim_provider_df = (
    providers_src
    .select(
        F.xxhash64("Id").alias("provider_key"),
        F.col("Id").alias("provider_id"),
        F.col("ORGANIZATION").alias("organization_id"),
        F.col("NAME").alias("provider_name"),
        F.col("GENDER").alias("gender"),
        F.col("SPECIALITY").alias("specialty"),
        F.col("ADDRESS").alias("address"),
        F.col("CITY").alias("city"),
        F.col("STATE").alias("state"),
        F.col("ZIP").alias("zip_code"),
        F.col("LAT").cast("double").alias("latitude"),
        F.col("LON").cast("double").alias("longitude"),
        F.col("UTILIZATION")
            .cast("long")
            .alias("utilization"),
    )
    .dropDuplicates(["provider_id"])
)

write_external_delta(
    dim_provider_df,
    TABLE_NAMES["provider"],
    TABLE_LOCATIONS["provider"],
)

print(f"Built {TABLE_NAMES['provider']}")

## 10. Build `dim_payer`

In [ ]:
payers_src = read_reference_csv("payers.csv")

# Synthea's payer schema uses STATE_HEADQUARTERED rather than STATE.
payer_state_source = (
    "STATE_HEADQUARTERED"
    if "STATE_HEADQUARTERED" in payers_src.columns
    else "STATE"
)

dim_payer_df = (
    payers_src
    .select(
        F.xxhash64("Id").alias("payer_key"),
        F.col("Id").alias("payer_id"),
        F.col("NAME").alias("payer_name"),
        F.col("ADDRESS").alias("address"),
        F.col("CITY").alias("city"),
        F.col(payer_state_source).alias("state"),
        F.col("ZIP").alias("zip_code"),
        F.col("PHONE").alias("phone"),
        F.col("AMOUNT_COVERED")
            .cast("decimal(18,2)")
            .alias("amount_covered"),
        F.col("AMOUNT_UNCOVERED")
            .cast("decimal(18,2)")
            .alias("amount_uncovered"),
        F.col("REVENUE")
            .cast("decimal(18,2)")
            .alias("revenue"),
        F.col("COVERED_ENCOUNTERS")
            .cast("long")
            .alias("covered_encounters"),
        F.col("UNCOVERED_ENCOUNTERS")
            .cast("long")
            .alias("uncovered_encounters"),
        F.col("UNIQUE_CUSTOMERS")
            .cast("long")
            .alias("unique_customers"),
        F.col("QOLS_AVG")
            .cast("double")
            .alias("qols_avg"),
        F.col("MEMBER_MONTHS")
            .cast("long")
            .alias("member_months"),
    )
    .dropDuplicates(["payer_id"])
)

write_external_delta(
    dim_payer_df,
    TABLE_NAMES["payer"],
    TABLE_LOCATIONS["payer"],
)

print(f"Built {TABLE_NAMES['payer']}")

## 11. Build `dim_condition`

In [ ]:
conditions_src = read_reference_csv("conditions.csv")

dim_condition_df = (
    conditions_src
    .select(
        F.col("CODE").alias("condition_code"),
        F.col("DESCRIPTION").alias("condition_description"),
    )
    .dropDuplicates(
        ["condition_code", "condition_description"]
    )
    .withColumn(
        "condition_key",
        F.xxhash64(
            "condition_code",
            "condition_description",
        ),
    )
    .select(
        "condition_key",
        "condition_code",
        "condition_description",
    )
)

write_external_delta(
    dim_condition_df,
    TABLE_NAMES["condition"],
    TABLE_LOCATIONS["condition"],
)

print(f"Built {TABLE_NAMES['condition']}")

## 12. Validate table existence, locations, and external registration

In [ ]:
registration_rows = []
registration_failures = []

for logical_name, table_name in TABLE_NAMES.items():
    exists = spark.catalog.tableExists(table_name)
    actual_location = (
        registered_table_location(table_name)
        if exists
        else None
    )
    expected_location = TABLE_LOCATIONS[logical_name]

    location_matches = (
        exists
        and actual_location is not None
        and normalize_location(actual_location)
        == normalize_location(expected_location)
    )

    registration_rows.append(
        (
            logical_name,
            table_name,
            expected_location,
            actual_location,
            "PASS" if location_matches else "FAIL",
        )
    )

    if not location_matches:
        registration_failures.append(logical_name)

registration_df = spark.createDataFrame(
    registration_rows,
    [
        "dimension",
        "table_name",
        "expected_location",
        "actual_location",
        "status",
    ],
)

display(registration_df)

if registration_failures:
    raise RuntimeError(
        "External-table registration failed for: "
        + ", ".join(registration_failures)
    )

## 13. Validate surrogate keys and dimension grains

In [ ]:
KEY_COLUMNS = {
    "date": "date_key",
    "patient": "patient_key",
    "provider": "provider_key",
    "organization": "organization_key",
    "payer": "payer_key",
    "condition": "condition_key",
}

key_rows = []
key_failures = []

for logical_name, key_column in KEY_COLUMNS.items():
    table_name = TABLE_NAMES[logical_name]
    row_count, distinct_keys, null_keys = dimension_key_profile(
        table_name,
        key_column,
    )

    status = (
        "PASS"
        if row_count > 0
        and row_count == distinct_keys
        and null_keys == 0
        else "FAIL"
    )

    key_rows.append(
        (
            logical_name,
            row_count,
            distinct_keys,
            null_keys,
            status,
        )
    )

    if status == "FAIL":
        key_failures.append(logical_name)

key_validation_df = spark.createDataFrame(
    key_rows,
    [
        "dimension",
        "row_count",
        "distinct_key_count",
        "null_key_count",
        "status",
    ],
)

display(key_validation_df)

if run_validation and key_failures:
    raise RuntimeError(
        "Dimension key validation failed for: "
        + ", ".join(key_failures)
    )

## 14. Validate important business relationships

In [ ]:
provider_org_missing = (
    spark.table(TABLE_NAMES["provider"]).alias("p")
    .join(
        spark.table(TABLE_NAMES["organization"])
        .select(
            F.col("organization_id").alias("_organization_id")
        )
        .dropDuplicates(),
        F.col("p.organization_id") == F.col("_organization_id"),
        "left_anti",
    )
    .filter(
        F.col("organization_id").isNotNull()
        & (F.trim(F.col("organization_id")) != "")
    )
    .count()
)

relationship_df = spark.createDataFrame(
    [
        (
            "provider_to_organization",
            provider_org_missing,
            "PASS" if provider_org_missing == 0 else "WARN",
        )
    ],
    ["relationship", "unmatched_rows", "status"],
)

display(relationship_df)

print(
    "Relationship mismatches are reported as WARN because source master "
    "data can contain retired or unmatched references."
)

## 15. Final validation summary

In [ ]:
final_checks = [
    ("source_external_volume", volume_type == "EXTERNAL"),
    ("reference_files", len(missing_reference_files) == 0),
    ("external_registration", len(registration_failures) == 0),
    ("dimension_keys", len(key_failures) == 0),
]

final_validation_df = spark.createDataFrame(
    [
        (
            check_name,
            "PASS" if passed else "FAIL",
        )
        for check_name, passed in final_checks
    ],
    ["validation_area", "status"],
)

display(final_validation_df)

failed_checks = [
    name
    for name, passed in final_checks
    if not passed
]

if run_validation and failed_checks:
    raise RuntimeError(
        "Lab 06 external dimension validation failed: "
        + ", ".join(failed_checks)
    )

print("")
print("LAB 06 EXTERNAL V2 — GOLD DIMENSIONS COMPLETE")
print(f"Target schema      : {target_schema_fqn}")
print(f"External Gold root : {external_gold_root}")
print("")
print("Created / validated:")
for logical_name, table_name in TABLE_NAMES.items():
    print(f"  - {table_name}")
print("")
print("Serverless compatibility: PASS")
print("REFRESH TABLE calls: 0")
print("")
print("Next: lab06_02_fact_encounters")

## Expected external layout

```text
<external_gold_root>/
├── dim_date/
├── dim_patient/
├── dim_provider/
├── dim_organization/
├── dim_payer/
└── dim_condition/
```

Each folder is a Delta table location registered in:

```text
<catalog>.<target_schema>
```

This preserves the Lab 06 external-table architecture while avoiding
`REFRESH TABLE`, which caused the Serverless failure in the previous notebook.
